In [5]:
import os
import random
import pandas as pd
import json
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend

SEQ_LEN = 256
NUM_SAMPLES_PER_MODE = 2000

def get_texts(num_texts):
    print("Loading dataset from CSV...")
    try:
        df = pd.read_csv('news_summary.csv', encoding='latin-1')
        raw_texts = df['text'].dropna().head(num_texts).tolist()
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return []

    texts = []
    for text in raw_texts:
        text_bytes = text.encode('utf-8')
        if len(text_bytes) >= SEQ_LEN:
            texts.append(text_bytes[:SEQ_LEN])
        else:
            padded_bytes = text_bytes + b' ' * (SEQ_LEN - len(text_bytes))
            texts.append(padded_bytes)

    return texts

def encrypt_aes(plaintext, mode_name):
    key = os.urandom(32)
    iv = os.urandom(16)

    if mode_name == 'CFB':
        mode = modes.CFB(iv)
    elif mode_name == 'OFB':
        mode = modes.OFB(iv)
    elif mode_name == 'CTR':
        mode = modes.CTR(iv)
    else:
        raise ValueError("Invalid mode")

    cipher = Cipher(algorithms.AES(key), mode, backend=default_backend())
    encryptor = cipher.encryptor()
    ciphertext = encryptor.update(plaintext) + encryptor.finalize()
    return ciphertext

def generate_data():
    texts = get_texts(NUM_SAMPLES_PER_MODE * 3)

    if not texts:
        print("No texts loaded. Exiting...")
        return

    modes_list = ['CFB', 'OFB', 'CTR']
    data = []

    print("Generating ciphertexts...")
    for i, mode in enumerate(modes_list):
        for j in range(NUM_SAMPLES_PER_MODE):
            idx = i * NUM_SAMPLES_PER_MODE + j

            if idx >= len(texts):
                break

            plaintext = texts[idx]
            ciphertext = encrypt_aes(plaintext, mode)
            data.append({
                'ciphertext': list(ciphertext),
                'mode': mode
            })

    random.shuffle(data)

    with open('ciphertexts.json', 'w') as f:
        json.dump(data, f)
    print(f"Generated {len(data)} samples saved to ciphertexts.json")

if __name__ == "__main__":
    generate_data()

Loading dataset from CSV...
Generating ciphertexts...


/var/folders/_p/d06km2k91v3753mkftwkfnqm0000gn/T/ipykernel_21500/1495027399.py:36: CryptographyDeprecationWarning: CFB has been moved to cryptography.hazmat.decrepit.ciphers.modes.CFB and will be removed from cryptography.hazmat.primitives.ciphers.modes in 49.0.0.
  mode = modes.CFB(iv)
/var/folders/_p/d06km2k91v3753mkftwkfnqm0000gn/T/ipykernel_21500/1495027399.py:38: CryptographyDeprecationWarning: OFB has been moved to cryptography.hazmat.decrepit.ciphers.modes.OFB and will be removed from cryptography.hazmat.primitives.ciphers.modes in 49.0.0.
  mode = modes.OFB(iv)


Generated 4514 samples saved to ciphertexts.json


In [6]:
import json
import numpy as np

def load_data():
    with open('ciphertexts.json', 'r') as f:
        data = json.load(f)
    return data

def differential_preprocess(ciphertext):
    """
    Computes the XOR difference between consecutive bytes of the ciphertext.
    Returns a sequence of length len(ciphertext) - 1.
    """
    diffs = []
    for i in range(1, len(ciphertext)):
        diffs.append(ciphertext[i] ^ ciphertext[i-1])
    return diffs

def preprocess_all_data():
    data = load_data()
    processed_data = []
    for item in data:
        ct = item['ciphertext']
        diffs = differential_preprocess(ct)
        processed_data.append({
            'features': diffs,
            'mode': item['mode']
        })
    
    with open('processed_data.json', 'w') as f:
        json.dump(processed_data, f)
    print(f"Preprocessed {len(processed_data)} samples saved to processed_data.json")

if __name__ == "__main__":
    preprocess_all_data()


Preprocessed 4514 samples saved to processed_data.json


In [7]:
import json
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf

def train_and_evaluate_tf_lstm(mode1, mode2, all_data):
    print(f"\n--- LSTM Classifier for {mode1} vs {mode2} ---")

    
    filtered_data = [d for d in all_data if d['mode'] in (mode1, mode2)]
    
    
    X = np.array([d['features'] for d in filtered_data], dtype=np.float32)
    X = X / 255.0
    
    
    X = np.expand_dims(X, axis=-1)

    
    y = np.array([0 if d['mode'] == mode1 else 1 for d in filtered_data], dtype=np.int32)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=(X_train.shape[1], 1)),
        tf.keras.layers.LSTM(64),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(2, activation='softmax')
    ])

    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

   
    model.fit(X_train, y_train, epochs=10, batch_size=64, verbose=0)
    
    
    predictions = model.predict(X_test, verbose=0)
    preds = np.argmax(predictions, axis=1)

    acc = accuracy_score(y_test, preds)
    cm = confusion_matrix(y_test, preds)

    print(f"Test Accuracy ({mode1} vs {mode2}): {acc*100:.2f}%")
    print("Confusion Matrix:\n", cm)
    return acc

if __name__ == "__main__":
    try:
        with open('processed_data.json', 'r') as f:
            all_data = json.load(f)

        acc1 = train_and_evaluate_tf_lstm('CFB', 'OFB', all_data)
        acc2 = train_and_evaluate_tf_lstm('CFB', 'CTR', all_data)
        acc3 = train_and_evaluate_tf_lstm('OFB', 'CTR', all_data)

        print("\n--- Summary ---")
        print(f"CFB vs OFB Accuracy: {acc1*100:.2f}%")
        print(f"CFB vs CTR Accuracy: {acc2*100:.2f}%")
        print(f"OFB vs CTR Accuracy: {acc3*100:.2f}%")

    except FileNotFoundError:
        print("Error: processed_data.json not found. Please run your data generator first.")


--- LSTM Classifier for CFB vs OFB ---


/Users/suyashsinha/Desktop/Classifier/venv/lib/python3.13/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Test Accuracy (CFB vs OFB): 48.12%
Confusion Matrix:
 [[385   0]
 [415   0]]

--- LSTM Classifier for CFB vs CTR ---


/Users/suyashsinha/Desktop/Classifier/venv/lib/python3.13/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Test Accuracy (CFB vs CTR): 79.52%
Confusion Matrix:
 [[400   0]
 [103   0]]

--- LSTM Classifier for OFB vs CTR ---


/Users/suyashsinha/Desktop/Classifier/venv/lib/python3.13/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Test Accuracy (OFB vs CTR): 78.93%
Confusion Matrix:
 [[397   0]
 [106   0]]

--- Summary ---
CFB vs OFB Accuracy: 48.12%
CFB vs CTR Accuracy: 79.52%
OFB vs CTR Accuracy: 78.93%
